# 03 — Tools: File Operations

**Module 2 of the workshop.** The **vended tools** demo — pre-built capabilities Strands ships, ready to import and attach, no implementation required.


## Problem

An LLM alone can't read a file, call an API, or run code. Tools are the mechanism through which an agent moves beyond text generation into acting on external systems — here, the local filesystem.


## Concept

Four mechanisms, one interface — the agent doesn't care which kind of tool it's calling:

```
                 TOOLS
                   │
       ┌───────────┼────────────┐
       │           │            │
     Custom      Vended        MCP
      Tools       Tools        Tools
       │           │            │
       └───────────┼────────────┘
                   │
             Agent-as-tool
```

This script uses the **vended** branch — `file_read`, `file_write`, `editor` are pre-built, imported straight from `strands_tools`, not written by you. **What happens internally:** the model decides "I need `file_write`" → emits a structured tool call → Strands executes it → result goes back into the conversation → model continues. The model never touches the filesystem itself, it only ever asks Strands to.


## Architecture

```
"Create notes.txt with 'Meeting notes'"
              │
              ▼
        ┌───────────┐
        │  Agent    │
        │ (model)   │
        └─────┬─────┘
              │ decides: file_write
              ▼
     ┌──────────────────┐
     │  file_write        │  (vended tool)
     │  (strands_tools)   │
     └────────┬───────────┘
              │ writes notes.txt to disk
              ▼
        confirmation ──────────▶ Agent replies

"Read notes.txt"
              │
              ▼
        ┌───────────┐
        │  Agent    │
        └─────┬─────┘
              │ decides: file_read
              ▼
     ┌──────────────────┐
     │  file_read         │  (vended tool)
     └────────┬───────────┘
              │ reads notes.txt from disk
              ▼
      file contents ──────────▶ Agent replies
```

Two separate calls, two separate tool decisions — the agent picks the right tool each time based on the request, not a hardcoded if/else you wrote.


## Step 1 — Windows gotcha: set BYPASS_TOOL_CONSENT before importing

`editor`/`file_write` normally ask for interactive y/n confirmation before touching a file — that hangs under most terminals (including Windows). Set this **before** importing `strands_tools`, not after.


In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("BYPASS_TOOL_CONSENT", "true")  # skip interactive y/n prompt (breaks under this shell)
sys.path.insert(0, str(Path.cwd().parent))


## Step 2 — Resolve the model and import the vended tools

`file_read`, `file_write`, `editor` come straight from `strands_tools` — no custom `@tool` function needed for this pattern.


In [2]:
from model_provider import get_model
from strands import Agent
from strands_tools import editor, file_read, file_write

model = get_model()
print(f"Using: {type(model).__name__}")


Using: OllamaModel


## Step 3 — Write the system prompt and build the agent


In [3]:
FILE_SYSTEM_PROMPT = """You are a file operations specialist. You help users read,
write, search, and modify files. Focus on providing clear information about file
operations and always confirm when files have been modified.
Key Capabilities:
1. Read files with various options (full content, line ranges, search)
2. Create and write to files
3. Edit existing files with precision
4. Report file information and statistics

Always specify the full file path in your responses for clarity."""

file_agent = Agent(
    model=model,
    system_prompt=FILE_SYSTEM_PROMPT,
    tools=[file_read, file_write, editor],
)


## Step 4 — Run it: write, then read

Two separate agent calls — watch the model pick a different tool (`file_write`, then `file_read`) for each, based on what the request actually needs.


In [4]:
file_agent("Create a new file called notes.txt with content 'Meeting notes' with dummy meetings notes for the next 3 days.")
file_agent("Read the contents of notes.txt")



Tool #1: file_write


╔═ File Write Operation ═╗
║                        ║
║ Path: notes.txt        ║
║ Size: 2502 characters  ║
║                        ║
╚════════════════════════╝

╔═══════════ Write Successful ═══════════╗
║ File written successfully to notes.txt ║
╚════════════════════════════════════════╝

I've successfully created the `notes.txt` file with dummy meeting notes for the next 3 days (Monday through Wednesday). 

The file contains:

**Day 1 (Monday):** Project management meeting covering sprint planning and resource allocation, with action items for documentation and stakeholder coordination.

**Day 2 (Tuesday):** Design team virtual meeting focused on UI/UX review and accessibility compliance, with updates on design specifications and mobile-first approach.

**Day 3 (Wednesday):** Operations meeting about supply chain status, vendor partnerships, and logistics optimization, requiring budget approval for software upgrades.

Each day includes attendees, agenda items, key discussion points, and action items for better structure and realism.
Tool #2: file_read


╔══════════════════════════════════════ 📄 notes.txt ═══════════════════════════════════════╗
║                                                                                           ║
║     1 Meeting notes                                                                       ║
║     2                                                                                     ║
║     3 Day 1 - Monday, October 16, 2024                                                    ║
║     4 ==================================                                                  ║
║     5 Time: 9:00 AM - 10:30 AM | Location: Conference Room A                              ║
║     6                                                                                     ║
║     7 Attendees:                                                                          ║
║     8 - Jane Smith (Project Manager)                                                      ║
║     9 - John Doe (Development Lead)                                                       ║
║    10 - Sarah Johnson (QA Specialist)                                                     ║
║    11                                                                                     ║
║    12 Agenda Items:                                                                       ║
║    13 1. Project status review                                                            ║
║    14 2. Sprint planning discussion                                                       ║
║    15 3. Resource allocation for Q4                                                       ║
║    16                                                                                     ║
║    17 Key Points Discussed:                                                               ║
║    18 - Development team is on track with the 3 backend modules complete                  ║
║    19 - QA needs to be ramped up by Friday to meet testing timeline                       ║
║    20 - Sarah will take ownership of integration test scheduling                          ║
║    21 - Jane requested additional support from marketing for launch coordination          ║
║    22                                                                                     ║
║    23 Action Items:                                                                       ║
║    24 - John to finalize API documentation by EOD tomorrow                                ║
║    25 - Jane to coordinate with stakeholders on Monday's all-hands meeting                ║
║    26 - Schedule follow-up review meeting for Tuesday at 2 PM                             ║
║    27                                                                                     ║
║    28 Day 2 - Tuesday, October 17, 2024                                                   ║
║    29 ==================================                                                  ║
║    30 Time: 2:00 PM - 3:30 PM | Location: Virtual Meeting (Zoom)                          ║
║    31                                                                                     ║
║    32 Attendees:                                                                          ║
║    33 - Mike Wilson (Design Team Lead)                                                    ║
║    34 - Emily Davis (Frontend Developer)                                                  ║
║    35 - Alex Chen (UX Researcher)                                                         ║
║    36                                                                                     ║
║    37 Agenda Items:                                                                       ║
║    38 1. UI/UX design review                                                              ║
║    39 2. Accessibility compliance discussion                                              ║
║    40 3. Mobile responsiveness updates                                                    ║
║    41                                              

I've successfully read the contents of `notes.txt`. Here's a summary of what's in the file:

**The file contains 3 days of meeting notes:**

📅 **Day 1 - Monday (Oct 16):** Project management meeting with Jane Smith, John Doe, and Sarah Johnson discussing sprint planning and resource allocation.

📅 **Day 2 - Tuesday (Oct 17):** Design team virtual meeting with Mike Wilson, Emily Davis, and Alex Chen focusing on UI/UX design review and accessibility compliance.

📅 **Day 3 - Wednesday (Oct 18):** Operations meeting with Jessica Brown, David Lee, and Rachel Green covering supply chain status and vendor partnerships.

Each day includes:
- Time and location details
- Attendee list with roles
- Agenda items
- Key discussion points
- Action items with deadlines

The notes are well-structured and contain realistic meeting content that could be used for actual project documentation or as templates for future meetings.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I've successfully read the contents of `notes.txt`. Here's a summary of what's in the file:\n\n**The file contains 3 days of meeting notes:**\n\n📅 **Day 1 - Monday (Oct 16):** Project management meeting with Jane Smith, John Doe, and Sarah Johnson discussing sprint planning and resource allocation.\n\n📅 **Day 2 - Tuesday (Oct 17):** Design team virtual meeting with Mike Wilson, Emily Davis, and Alex Chen focusing on UI/UX design review and accessibility compliance.\n\n📅 **Day 3 - Wednesday (Oct 18):** Operations meeting with Jessica Brown, David Lee, and Rachel Green covering supply chain status and vendor partnerships.\n\nEach day includes:\n- Time and location details\n- Attendee list with roles\n- Agenda items\n- Key discussion points\n- Action items with deadlines\n\nThe notes are well-structured and contain realistic meeting content that could be used for actual project documentation or as temp

## Failure mode to know about

A tool with an ambiguous docstring gets called at the wrong time, or not at all — the model routes on the tool's *description*, not its implementation. A bad tool description is a bug, even if the function itself is correct. This matters even for vended tools: if you ever wrap or rename one, keep the docstring accurate to how you actually want it invoked.
